# 02 — Build the First Pandera Schema

## Real-world problem

You already inspected the dirty order CSV.

Now the analytics team wants an **executable data contract** that can reject:

- duplicate order IDs
- non-positive quantities
- non-positive unit prices
- discounts outside `[0, 1]`
- unsupported order statuses
- incorrect column dtypes

This notebook implements **Phase 2 only**.

We deliberately do **not** solve coercion, invalid dates, lazy validation, or the `total` cross-column rule yet.

## 1. Project setup

Run Jupyter from the repository root when possible.

The following cell also works if the notebook kernel starts inside `notebooks/`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ROOT

WindowsPath('C:/Users/Victus 16/PycharmProjects/pandera-data-quality-lab')

In [2]:
import pandas as pd
import pandera.pandas as pa

from pandera_lab.schemas import OrderSchema

## 2. Inspect the executable contract

`DataFrameModel.to_schema()` lets us inspect the object-based schema generated behind the class-based model.

In [3]:
OrderSchema.to_schema()

<Schema DataFrameSchema(columns={'order_id': <Schema Column(name=order_id, type=DataType(int64))>, 'customer_id': <Schema Column(name=customer_id, type=DataType(str))>, 'product_id': <Schema Column(name=product_id, type=DataType(str))>, 'quantity': <Schema Column(name=quantity, type=DataType(int64))>, 'unit_price': <Schema Column(name=unit_price, type=DataType(float64))>, 'discount': <Schema Column(name=discount, type=DataType(float64))>, 'total': <Schema Column(name=total, type=DataType(float64))>, 'status': <Schema Column(name=status, type=DataType(str))>, 'order_date': <Schema Column(name=order_date, type=DataType(datetime64[ns]))>}, checks=[], parsers=[], index=None, dtype=None, coerce=False, strict=False, name=OrderSchema, ordered=False, unique=None, report_duplicates=all, unique_column_names=False, add_missing_columns=False, title=None, description=Column-level data contract for an analytical order record., metadata=None, drop_invalid_rows=False)>

## 3. Start with known-good data

The reference CSV contains rows that should satisfy the Phase-2 rules.

Notice that we explicitly parse `order_date` here. Parsing raw dates will become part of the real ingestion design in Phase 3.

In [4]:
reference_path = ROOT / "data" / "reference" / "orders_valid.csv"

reference_df = pd.read_csv(
    reference_path,
    parse_dates=["order_date"],
)

reference_df

,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date
0,2001,C101,P101,2,100.0,0.10,180.0,paid,2026-08-01
1,2002,C102,P102,1,50.0,0.00,50.0,pending,2026-08-02
2,2003,C103,P103,3,20.0,0.25,45.0,shipped,2026-08-03
3,2004,C104,P104,1,250.0,0.20,200.0,cancelled,2026-08-04
4,2005,C105,P105,4,15.0,0.00,60.0,paid,2026-08-05


In [5]:
reference_df.dtypes

order_id                int64
customer_id            object
product_id             object
quantity                int64
unit_price            float64
discount              float64
total                 float64
status                 object
order_date     datetime64[ns]
dtype: object

In [6]:
validated_reference = OrderSchema.validate(reference_df)
print("Phase-2 reference data is valid ✅")
validated_reference

Phase-2 reference data is valid ✅


,order_id,customer_id,product_id,quantity,unit_price,discount,total,status,order_date
0,2001,C101,P101,2,100.0,0.10,180.0,paid,2026-08-01
1,2002,C102,P102,1,50.0,0.00,50.0,pending,2026-08-02
2,2003,C103,P103,3,20.0,0.25,45.0,shipped,2026-08-03
3,2004,C104,P104,1,250.0,0.20,200.0,cancelled,2026-08-04
4,2005,C105,P105,4,15.0,0.00,60.0,paid,2026-08-05


## 4. A tiny helper for experiments

For now we use normal fail-fast validation.

Phase 3 introduces `lazy=True` and structured failure reports.

In [7]:
def try_validate(name: str, dataframe: pd.DataFrame) -> None:
    print(f"\n--- {name} ---")
    try:
        OrderSchema.validate(dataframe)
        print("PASS ✅")
    except pa.errors.SchemaError as exc:
        print("FAIL ❌")
        print(str(exc).splitlines()[-1])

## 5. Uniqueness rule

Two records cannot have the same `order_id`.

In [8]:
duplicate_id = reference_df.copy()
duplicate_id.loc[1, "order_id"] = duplicate_id.loc[0, "order_id"]

try_validate("duplicate order_id", duplicate_id)


--- duplicate order_id ---
FAIL ❌
Name: order_id, dtype: int64


## 6. Numeric range rules

`quantity` must be greater than zero.

In [9]:
zero_quantity = reference_df.copy()
zero_quantity.loc[0, "quantity"] = 0

try_validate("zero quantity", zero_quantity)


--- zero quantity ---
FAIL ❌
Column 'quantity' failed element-wise validator number 0: greater_than(0) failure cases: 0


`unit_price` must also be greater than zero.

In [10]:
negative_price = reference_df.copy()
negative_price.loc[0, "unit_price"] = -10.0

try_validate("negative unit_price", negative_price)


--- negative unit_price ---
FAIL ❌
Column 'unit_price' failed element-wise validator number 0: greater_than(0) failure cases: -10.0


`discount` must stay inside the closed interval `[0, 1]`.

In [11]:
bad_discount = reference_df.copy()
bad_discount.loc[0, "discount"] = 1.20

try_validate("discount > 1", bad_discount)


--- discount > 1 ---
FAIL ❌
Column 'discount' failed element-wise validator number 1: less_than_or_equal_to(1) failure cases: 1.2


## 7. Allowed categorical values

A string can have the correct dtype and still violate the business domain.

In [12]:
bad_status = reference_df.copy()
bad_status.loc[0, "status"] = "refunded"

try_validate("unsupported status", bad_status)


--- unsupported status ---
FAIL ❌
Column 'status' failed element-wise validator number 0: isin(('pending', 'paid', 'shipped', 'cancelled')) failure cases: refunded


## 8. Dtype rule

Phase 2 expects `order_date` to already be a datetime column.

We intentionally do **not** coerce strings yet.

In [13]:
string_date = reference_df.copy()
string_date["order_date"] = string_date["order_date"].dt.strftime("%Y-%m-%d")

print(string_date["order_date"].dtype)
try_validate("string order_date", string_date)

object

--- string order_date ---
FAIL ❌
expected series 'order_date' to have type datetime64[ns], got object


## 9. Important limitation: wrong totals still pass

The following row is logically wrong, but all individual columns satisfy the Phase-2 rules.

This is why cross-column validation exists.

In [14]:
wrong_total = reference_df.copy()
wrong_total.loc[0, "total"] = 999999.0

try_validate("wrong total (expected to pass in Phase 2)", wrong_total)


--- wrong total (expected to pass in Phase 2) ---
PASS ✅


### Why did it pass?

Because Phase 2 only says:

```text
total is a float
```

It does not yet say:

```text
total == unit_price * quantity * (1 - discount)
```

That will become a dataframe-level business rule in Phase 4.

## 10. Extra columns are temporarily allowed

Our current design uses `strict=False`.

So an extra operational column does not fail validation yet.

In [15]:
extra_column = reference_df.copy()
extra_column["internal_note"] = "temporary operational metadata"

validated = OrderSchema.validate(extra_column)
print("internal_note" in validated.columns)

True


## 11. Try the raw CSV

Now load the deliberately dirty file **without** fixing it.

Do not expect Phase 2 to produce a clean error report yet.

The raw CSV contains ingestion problems that motivate Phase 3.

In [16]:
raw_path = ROOT / "data" / "raw" / "orders.csv"
raw_df = pd.read_csv(raw_path)

raw_df.dtypes

order_id           int64
customer_id       object
product_id        object
quantity          object
unit_price       float64
discount         float64
total            float64
status            object
order_date        object
internal_note     object
dtype: object

In [17]:
try_validate("raw orders.csv", raw_df)


--- raw orders.csv ---
FAIL ❌
Name: order_id, dtype: int64


# Phase-2 checkpoint

You should now be able to explain:

1. why `Series[int]` and `gt=0` solve different problems,
2. why `unique=True` is not a dtype rule,
3. why `isin` is useful for business domains,
4. why a wrong `total` still passes,
5. why raw CSV parsing should not be mixed blindly into this phase.

## Next phase

Phase 3 will solve the messy-input boundary:

- coercion
- nullable values
- date parsing
- `lazy=True`
- `failure_cases`
- deciding what to do with extra columns